# Hand Gesture Recognition Model Training

This notebook demonstrates how to train a gesture classification model using hand landmarks extracted from MediaPipe. The model will learn to classify different hand gestures (e.g., pinch, open hand) based on landmark coordinates.

**Prerequisites:**
- Webcam for data collection
- MediaPipe hand landmarker model (hand_landmarker.task)
- Python packages: tensorflow, scikit-learn, opencv, mediapipe


## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import tensorflow as tf
from tensorflow import keras
from keras import layers, Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle
from pathlib import Path

print("✓ All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"MediaPipe version: {mp.__version__}")

## 2. Load and Prepare Data

### Data Collection from Webcam
First, we'll collect hand gesture samples from the webcam and extract landmarks.

Set `COLLECT_DATA = True` to gather training samples. Press keys 1-3 for different gestures:

In [ ]:
# ─── Configuration ─────────────────────────────────────────────────────────
COLLECT_DATA = False  # Set to True if you want to collect new gesture data
DATA_DIR = "gesture_data"
GESTURE_NAMES = ["pinch", "open", "peace"]  # Define your gesture classes
NUM_LANDMARKS = 21  # MediaPipe hand has 21 landmarks

# Create data directory if it doesn't exist
Path(DATA_DIR).mkdir(exist_ok=True)

# ─── Load MediaPipe Hand Landmarker ────────────────────────────────────────
base_options = python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.HandLandmarker.create_from_options(options)

# ─── Data Collection Function ──────────────────────────────────────────────
def collect_gesture_data():
    """
    Collect hand gesture samples from webcam.
    Press 1, 2, 3 for different gestures. Press 'q' to quit.
    """
    webcam = cv2.VideoCapture(0)
    gesture_samples = {gesture: [] for gesture in GESTURE_NAMES}
    
    print(f"Collecting data for gestures: {GESTURE_NAMES}")
    print("Controls: 1=pinch, 2=open, 3=peace, q=quit, s=save")
    
    while True:
        ret, frame = webcam.read()
        if not ret:
            break
            
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        
        # Detect hand landmarks
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        result = detector.detect(mp_image)
        
        if result.hand_landmarks:
            landmarks = result.hand_landmarks[0]
            # Extract normalized coordinates
            landmark_list = np.array([[lm.x, lm.y, lm.z] for lm in landmarks]).flatten()
            
            # Display
            cv2.putText(frame, "Hand detected", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            for i, gesture in enumerate(GESTURE_NAMES):
                cv2.putText(frame, f"{i+1}: {gesture} ({len(gesture_samples[gesture])})", 
                           (10, 70 + i*30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1)
        else:
            landmark_list = None
            cv2.putText(frame, "No hand detected", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        
        cv2.imshow("Gesture Data Collection", frame)
        
        # Handle key presses
        key = cv2.waitKey(10) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('1') and landmark_list is not None:
            gesture_samples["pinch"].append(landmark_list)
            print(f"✓ Added pinch sample ({len(gesture_samples['pinch'])})")
        elif key == ord('2') and landmark_list is not None:
            gesture_samples["open"].append(landmark_list)
            print(f"✓ Added open sample ({len(gesture_samples['open'])})")
        elif key == ord('3') and landmark_list is not None:
            gesture_samples["peace"].append(landmark_list)
            print(f"✓ Added peace sample ({len(gesture_samples['peace'])})")
        elif key == ord('s'):
            # Save collected data
            for gesture, samples in gesture_samples.items():
                if samples:
                    np.save(f"{DATA_DIR}/{gesture}.npy", np.array(samples))
                    print(f"✓ Saved {len(samples)} {gesture} samples")
    
    webcam.release()
    cv2.destroyAllWindows()
    return gesture_samples

# Collect data if flag is set
if COLLECT_DATA:
    collected_data = collect_gesture_data()
else:
    print("⚠ Data collection skipped. Generating synthetic training data for demonstration...")

In [ ]:
# ─── Load Collected Data or Generate Synthetic Data ───────────────────────
X_data = []
y_data = []

# Try to load saved data
data_loaded = False
for gesture_idx, gesture in enumerate(GESTURE_NAMES):
    data_file = f"{DATA_DIR}/{gesture}.npy"
    if os.path.exists(data_file):
        samples = np.load(data_file)
        X_data.extend(samples)
        y_data.extend([gesture_idx] * len(samples))
        data_loaded = True
        print(f"✓ Loaded {len(samples)} {gesture} samples")

# If no real data, generate synthetic data for demonstration
if not data_loaded:
    print("📊 Generating synthetic training data for demonstration...")
    np.random.seed(42)
    
    for gesture_idx, gesture in enumerate(GESTURE_NAMES):
        # Generate 100 samples per gesture with slight variations
        num_samples = 100
        base_landmarks = np.random.randn(NUM_LANDMARKS * 3) * 0.1 + 0.5
        
        for _ in range(num_samples):
            # Add noise to create variations
            noise = np.random.randn(NUM_LANDMARKS * 3) * 0.05
            sample = base_landmarks + noise
            sample = np.clip(sample, 0, 1)  # Normalize to [0, 1]
            X_data.append(sample)
            y_data.append(gesture_idx)
    
    print(f"✓ Generated {len(X_data)} synthetic samples ({len(X_data) // len(GESTURE_NAMES)} per gesture)")

# Convert to numpy arrays
X = np.array(X_data)
y = np.array(y_data)

print(f"\n📋 Dataset Summary:")
print(f"  Shape: {X.shape}")
print(f"  Total samples: {len(X)}")
print(f"  Features (landmarks * 3): {X.shape[1]}")
print(f"  Classes: {GESTURE_NAMES}")
for i, gesture in enumerate(GESTURE_NAMES):
    count = np.sum(y == i)
    print(f"    {gesture}: {count} samples ({count/len(y)*100:.1f}%)")

## 3. Split Data into Training and Testing Sets

In [ ]:
# ─── Normalize features ────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ─── Split into training and testing sets ───────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Data Split Summary:")
print(f"  Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Testing set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"  Feature dimension: {X_train.shape[1]}")

# Verify class distribution
print(f"\n📊 Training set class distribution:")
for i, gesture in enumerate(GESTURE_NAMES):
    count = np.sum(y_train == i)
    print(f"    {gesture}: {count} samples ({count/len(y_train)*100:.1f}%)")

# Save scaler for inference
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✓ Scaler saved for inference")

## 4. Build the Model Architecture

We'll create a neural network with dropout layers to prevent overfitting:

In [ ]:
# ─── Build Neural Network Model ────────────────────────────────────────────
input_size = X_train.shape[1]
num_classes = len(GESTURE_NAMES)

model = Sequential([
    layers.Input(shape=(input_size,)),
    
    # First dense layer with ReLU activation
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Second dense layer
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Third dense layer
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    
    # Output layer with softmax for multi-class classification
    layers.Dense(num_classes, activation='softmax')
])

# Display model architecture
print("🏗️ Model Architecture:")
model.summary()

## 5. Compile the Model

In [ ]:
# ─── Compile the Model ─────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',  # For integer labels
    metrics=['accuracy']
)

print("✓ Model compiled successfully!")
print(f"  Optimizer: Adam (lr=0.001)")
print(f"  Loss: sparse_categorical_crossentropy")
print(f"  Metrics: accuracy")

## 6. Train the Model

Training with 50 epochs and validation split of 20%:

In [ ]:
# ─── Train the Model ───────────────────────────────────────────────────────
EPOCHS = 50
BATCH_SIZE = 32

print(f"🚀 Starting training for {EPOCHS} epochs...\n")

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,  # Use 20% of training data for validation
    verbose=1,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        )
    ]
)

print("\n✓ Training completed!")

## 7. Evaluate Model Performance

In [ ]:
# ─── Evaluate on Test Set ──────────────────────────────────────────────────
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"📊 Test Set Performance:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# ─── Predictions on Test Set ────────────────────────────────────────────────
y_pred = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

# ─── Classification Report ─────────────────────────────────────────────────
print(f"\n📈 Classification Report:")
print(classification_report(y_test, y_pred_classes, target_names=GESTURE_NAMES))

# ─── Confusion Matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_classes)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=GESTURE_NAMES, yticklabels=GESTURE_NAMES,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.show()

print("✓ Confusion matrix displayed")

In [ ]:
# ─── Plot Training History ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy During Training')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss During Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Training history plots displayed")

## 8. Save the Trained Model

Save the model for future use in the gesture control application:

In [ ]:
# ─── Save the Model ────────────────────────────────────────────────────────
model_path = 'gesture_classifier_model.h5'
model.save(model_path)

print(f"✓ Model saved to '{model_path}'")
print(f"  File size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")

# ─── Save Model Configuration ──────────────────────────────────────────────
config = {
    'gesture_names': GESTURE_NAMES,
    'input_size': input_size,
    'num_classes': num_classes,
    'model_path': model_path,
    'scaler_path': 'scaler.pkl',
    'test_accuracy': float(test_accuracy),
    'test_loss': float(test_loss),
}

import json
with open('model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Model configuration saved to 'model_config.json'")

print(f"\n🎉 Model training complete!")
print(f"   - Test Accuracy: {test_accuracy*100:.2f}%")
print(f"   - Model: {model_path}")
print(f"   - Config: model_config.json")
print(f"   - Scaler: scaler.pkl")

## Bonus: Load and Test the Model

Example of loading the trained model for inference:

In [ ]:
# ─── Load Saved Model for Inference ────────────────────────────────────────
loaded_model = keras.models.load_model('gesture_classifier_model.h5')

# Load configuration
with open('model_config.json', 'r') as f:
    model_config = json.load(f)

# Load scaler
with open('scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

print("✓ Model loaded successfully!")
print(f"  Gestures: {model_config['gesture_names']}")
print(f"  Test Accuracy: {model_config['test_accuracy']:.2%}")

# ─── Test on a Sample from Test Set ────────────────────────────────────────
sample_idx = 0
sample = X_test[sample_idx].reshape(1, -1)
true_label = y_test[sample_idx]

prediction = loaded_model.predict(sample, verbose=0)
predicted_class = np.argmax(prediction, axis=1)[0]
confidence = np.max(prediction[0])

print(f"\n🎯 Sample Prediction:")
print(f"  True gesture: {GESTURE_NAMES[true_label]}")
print(f"  Predicted gesture: {GESTURE_NAMES[predicted_class]}")
print(f"  Confidence: {confidence*100:.2f}%")
print(f"  All probabilities:")
for i, gesture in enumerate(GESTURE_NAMES):
    print(f"    {gesture}: {prediction[0][i]*100:.2f}%")

## Summary

This notebook provides a complete workflow for training a gesture classification model:

1. **Data Collection**: Capture hand gestures using MediaPipe landmark detection
2. **Data Preparation**: Extract landmarks and normalize features
3. **Model Architecture**: Build a 3-layer neural network with dropout regularization
4. **Training**: Train with early stopping and validation monitoring
5. **Evaluation**: Assess performance using accuracy, precision, recall, and F1-score
6. **Visualization**: Display confusion matrix and training history curves
7. **Model Export**: Save the trained model and configuration for deployment

### Next Steps:
- Collect more real training data for better performance
- Experiment with different architectures (CNN, LSTM for temporal sequences)
- Deploy the model in a real-time gesture control application
- Fine-tune hyperparameters for optimal accuracy

### Files Generated:
- `gesture_classifier_model.h5` - Trained model
- `model_config.json` - Model configuration
- `scaler.pkl` - Feature scaler for preprocessing
- `gesture_data/` - Directory for collected gesture samples